# U.S. and Canadian Census-Designated Places Road Network Analysis

**Overview:**

This code’s primary purpose is to analyze road network characteristics by linking the geographic boundaries of both U.S. Census-designated places and their Canadian equivalents to pre-saved road-network graphs (in GraphML format). It then computes and records various network statistics. The workflow is structured into several major steps:

1. **Reading and Preparing Data:**
   - **U.S. Data:**  
     The code locates directories for U.S. Census-designated places (organized by state or D.C.), loads the corresponding shapefiles, and applies a population-based filter to restrict the analysis to places below a specified population threshold.
   - **Canadian Data:**  
     Similarly, it reads the designated places shapefile for Canada, standardizes required columns (e.g., renaming fields like `DPLNAME` and `DPLUID`), and ensures the data is in the correct coordinate system (EPSG:4326).

2. **Associating Places With Road-Network Graphs:**
   - For each designated place (from the U.S. or Canada), the code attempts to load a corresponding road-network graph that was previously saved as a GraphML file.
   - The graph is then “clipped” or “truncated” using geographic operations to extract the portion of the road network within (or near) the place’s boundary—optionally expanded by a configurable buffer distance. This creates a local subset of the road network specific to the area of interest.

3. **Computing Network Statistics:**
   - With the place-specific network obtained, the code computes various metrics such as graph density, average node degree, and counts of edges that cross the place’s boundary.
   - It further breaks down these boundary-crossing counts by highway type (e.g., motorway, trunk, primary, secondary, tertiary), and aggregates associated attributes such as lane counts.
   - When enabled, the code also generates visualizations that overlay the clipped network on the place boundary and annotate key network statistics.

4. **Handling Errors and Timeouts:**
   - Each designated place is processed independently and in parallel using a `ProcessPoolExecutor` to speed up execution.
   - The code employs timeout mechanisms and robust error handling to catch issues (such as delays in loading or clipping large graphs) without stopping the overall processing. All timeouts, exceptions, and warnings are logged for later review.

5. **Collecting and Saving Results:**
   - All computed statistics from both U.S. and Canadian places are aggregated into a single DataFrame.
   - The final results are saved as a CSV file for further analysis, and diagnostic plots (such as histograms and network visualizations) are generated and stored.

## USA

In [ ]:
import os
WRI_PROJECT_ROOT = os.environ.get("WRI_PROJECT_ROOT", "/home/shares/wwri-wildfire")

import os
import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use a non-interactive backend
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, MultiLineString, Point
from matplotlib.lines import Line2D
import networkx as nx
import numpy as np
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial
from tqdm import tqdm
import logging
import glob
import signal
import ast
import multiprocessing
from collections import defaultdict

################################################################################
#                            CONFIGURATION / SETTINGS                          #
################################################################################
print(">>> Setting up configuration...")
NUM_CORES = 200
BUFFER_SIZES_ANALYSIS_METERS = [500]

# US-specific directory configurations
GRAPHML_OUTPUT_DIR_US = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'osmnx_road_network', '2024', 'US')
PLOTS_OUTPUT_DIR_US = os.path.join(WRI_PROJECT_ROOT, 'data', 'infrastructure', 'intermediate', 'road_network', '2024', 'us-network-plots')
CSV_OUTPUT_DIR_US = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'osmnx_road_network', '2024', 'US')
US_CENSUS_PLACES_DIR = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'raw', 'boundary_layers', 'us_census_designated_places', '2024')
FINAL_CSV_OUTPUT_US = os.path.join(WRI_PROJECT_ROOT, 'data', 'infrastructure', 'intermediate', 'road_network', '2024', 'us_roadnetwork_statistics.csv')

CREATE_PLOTS = True
TIMEOUT_SECONDS = 600
SELECTED_STATES = []  # Leave empty to process all states, or specify like: ['California', 'Texas']

os.makedirs(GRAPHML_OUTPUT_DIR_US, exist_ok=True)
os.makedirs(PLOTS_OUTPUT_DIR_US, exist_ok=True)
os.makedirs(CSV_OUTPUT_DIR_US, exist_ok=True)
print(">>> Configuration complete.")

################################################################################
#                                HELPER FUNCTIONS                              #
################################################################################
def sanitize_filename(name):
    return "".join(c if c.isalnum() else "_" for c in name).strip("_")

def get_exterior_boundary(polygon):
    if isinstance(polygon, Polygon):
        return polygon.exterior
    elif isinstance(polygon, MultiPolygon):
        return MultiLineString([p.exterior for p in polygon.geoms])
    else:
        raise ValueError("Unsupported geometry type for exterior boundary extraction.")

def sanitize_hw_type(hw_type):
    return hw_type.replace(' ', '_').replace('[', '').replace(']', '').replace(',', '').replace("'", "")

def timeout_handler(signum, frame):
    raise TimeoutError("Operation timed out inside worker process.")

def extract_points(geometry):
    points = []
    if geometry.is_empty:
        return points
    elif geometry.geom_type == 'Point':
        points.append(geometry)
    elif geometry.geom_type == 'MultiPoint':
        points.extend(list(geometry.geoms))
    elif geometry.geom_type == 'GeometryCollection':
        for geom in geometry.geoms:
            if geom.geom_type == 'Point':
                points.append(geom)
    elif geometry.geom_type == 'LineString':
        for coord in geometry.coords:
            points.append(Point(coord))
    return points

def find_graphml_file(graphml_dir, place):
    """
    Find the GraphML file for a given US place.
    Uses the state name and place name/GEOID to construct the path.
    Returns: (graph_path, place_folder_name, attempted_paths)
    """
    state_name = place['STATE_NAME']
    place_name = place['NAME']
    geoid = place['GEOID']
    
    sanitized_name = sanitize_filename(place_name)
    place_folder_name = f"{sanitized_name}_{geoid}"
    
    attempted_paths = []
    
    # Primary path attempt
    graph_path = os.path.join(graphml_dir, state_name, place_folder_name, f"{place_folder_name}.graphml")
    attempted_paths.append(graph_path)
    
    if os.path.exists(graph_path):
        return graph_path, place_folder_name, attempted_paths
    
    # Alternative: look for any .graphml file in the folder
    folder_path = os.path.join(graphml_dir, state_name, place_folder_name)
    attempted_paths.append(f"{folder_path}/*.graphml")
    
    if os.path.exists(folder_path):
        graphml_files = glob.glob(os.path.join(folder_path, "*.graphml"))
        if graphml_files:
            return graphml_files[0], place_folder_name, attempted_paths
        else:
            # Add the actual files found in the directory for debugging
            all_files = glob.glob(os.path.join(folder_path, "*"))
            attempted_paths.append(f"Files found in {folder_path}: {[os.path.basename(f) for f in all_files]}")
    else:
        # Check if the state directory exists
        state_dir = os.path.join(graphml_dir, state_name)
        if os.path.exists(state_dir):
            available_places = [d for d in os.listdir(state_dir) if os.path.isdir(os.path.join(state_dir, d))]
            attempted_paths.append(f"Available place directories in {state_name}: {available_places[:10]}...")  # Limit to first 10
        else:
            attempted_paths.append(f"State directory does not exist: {state_dir}")
            # List available state directories
            if os.path.exists(graphml_dir):
                available_states = [d for d in os.listdir(graphml_dir) if os.path.isdir(os.path.join(graphml_dir, d))]
                attempted_paths.append(f"Available state directories: {available_states}")
    
    return None, None, attempted_paths

################################################################################
#                              PROCESSING FUNCTION                             #
################################################################################
def process_place(place, graphml_dir, plots_dir, csv_dir, buffer_sizes_analysis):
    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(TIMEOUT_SECONDS)

    relevant_highway_types = ['motorway', 'trunk', 'primary', 'secondary', 'tertiary']
    color_map = {
        'motorway': 'red',
        'trunk': 'orange',
        'primary': 'green',
        'secondary': 'purple',
        'tertiary': 'blue',
        'residential': 'lightgray'
    }

    def is_relevant_road(hw):
        if isinstance(hw, str) and hw.strip().startswith('[') and hw.strip().endswith(']'):
            try:
                hw_list = ast.literal_eval(hw)
            except Exception:
                hw_list = [hw]
        elif isinstance(hw, str):
            hw_list = [hw]
        elif isinstance(hw, list):
            hw_list = hw
        else:
            hw_list = [str(hw)]
        return any(sanitize_hw_type(h) in relevant_highway_types for h in hw_list)

    stats_list = []
    error_counter = defaultdict(int)

    try:
        place_name = place['NAME']
        geoid = place['GEOID']
        state_name = place['STATE_NAME']
        polygon = place.geometry

        # Find the GraphML file
        graph_path, place_folder_name, attempted_paths = find_graphml_file(graphml_dir, place)

        if graph_path is None:
            error_msg = (
                f"GraphML file not found for {place_name} (GEOID: {geoid}) in state {state_name}. "
                f"Attempted paths: {'; '.join(attempted_paths)}"
            )
            error_counter[error_msg] += 1
            logging.error(error_msg)
            return stats_list, list(error_counter.keys())
        
        # Set up output directories
        plots_output_dir = os.path.join(plots_dir, state_name, place_folder_name)
        os.makedirs(plots_output_dir, exist_ok=True)

        # Use the new CSV output directory structure (matching Canada's approach)
        csv_output_dir = os.path.join(csv_dir, state_name, place_folder_name, 'csv')
        os.makedirs(csv_output_dir, exist_ok=True)

        # Load graph
        try:
            G = ox.load_graphml(graph_path)
        except Exception as e:
            error_msg = f"Failed to load GraphML file for {place_name} (GEOID: {geoid}): {e}. File path: {graph_path}"
            error_counter[error_msg] += 1
            logging.error(error_msg)
            return stats_list, list(error_counter.keys())

        # Check if graph has any edges before proceeding
        if G.number_of_edges() == 0:
            error_msg = f"Graph contains no edges for {place_name} (GEOID: {geoid}). File path: {graph_path}"
            error_counter[error_msg] += 1
            logging.warning(error_msg)
            for buffer_size_analysis in buffer_sizes_analysis:
                stats = place.drop(labels='geometry', errors='ignore').to_dict()
                stats.update({
                    'buffer_distance': buffer_size_analysis,
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
            return stats_list, list(error_counter.keys())

        # Convert to GeoDataFrames with error handling
        try:
            gdf_nodes_full, gdf_edges_full = ox.graph_to_gdfs(G, nodes=True, edges=True)
        except ValueError as e:
            if "Graph contains no edges" in str(e):
                error_msg = f"Graph contains no edges for {place_name} (GEOID: {geoid}). File path: {graph_path}"
                error_counter[error_msg] += 1
                logging.warning(error_msg)
                for buffer_size_analysis in buffer_sizes_analysis:
                    stats = place.drop(labels='geometry', errors='ignore').to_dict()
                    stats.update({
                        'buffer_distance': buffer_size_analysis,
                    })
                    stats_list.append(stats)
                    empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                    empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                return stats_list, list(error_counter.keys())
            else:
                raise e
        except Exception as e:
            error_msg = f"Failed to convert graph to GeoDataFrames for {place_name} (GEOID: {geoid}): {e}. File path: {graph_path}"
            error_counter[error_msg] += 1
            logging.error(error_msg)
            return stats_list, list(error_counter.keys())

        if gdf_edges_full is None or gdf_edges_full.empty:
            error_msg = f"Graph contains no edges for {place_name} (GEOID: {geoid}). File path: {graph_path}"
            error_counter[error_msg] += 1
            logging.warning(error_msg)
            for buffer_size_analysis in buffer_sizes_analysis:
                stats = place.drop(labels='geometry', errors='ignore').to_dict()
                stats.update({
                    'buffer_distance': buffer_size_analysis,
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
            return stats_list, list(error_counter.keys())

        # Ensure CRS is set to EPSG:4326
        if gdf_edges_full.crs is None:
            gdf_edges_full.set_crs(epsg=4326, inplace=True)
        elif gdf_edges_full.crs.to_string() != 'EPSG:4326':
            gdf_edges_full = gdf_edges_full.to_crs(epsg=4326)

        place_data = place.drop(labels='geometry', errors='ignore').to_dict()

        for buffer_size_analysis in buffer_sizes_analysis:
            stats = place_data.copy()
            stats['buffer_distance'] = buffer_size_analysis

            # Buffer the polygon
            polygon_series = gpd.GeoSeries([polygon], crs='EPSG:4326')
            utm_crs = polygon_series.estimate_utm_crs()
            polygon_utm = polygon_series.to_crs(utm_crs)
            polygon_buffered_analysis = polygon_utm.buffer(buffer_size_analysis).to_crs('EPSG:4326').iloc[0]

            # Clip the graph with additional error handling
            try:
                # Check if graph is empty before clipping
                if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
                    raise ValueError("Empty graph cannot be clipped")
                
                G_clipped = ox.truncate.truncate_graph_polygon(
                    G,
                    polygon_buffered_analysis,
                    truncate_by_edge=True
                )
            except ValueError as e:
                if "Empty graph cannot be clipped" in str(e):
                    error_msg = f"Cannot clip empty graph for {place_name} (GEOID: {geoid}) at {buffer_size_analysis}m. File path: {graph_path}"
                elif "No graph nodes found within buffer" in str(e) or "No graph nodes remain" in str(e):
                    error_msg = f"No graph nodes found within buffer for {place_name} (GEOID: {geoid}) at {buffer_size_analysis}m. File path: {graph_path}"
                else:
                    error_msg = f"Graph clipping failed for {place_name} (GEOID: {geoid}) at {buffer_size_analysis}m: {e}. File path: {graph_path}"
                error_counter[error_msg] += 1
                logging.warning(error_msg)
                stats.update({
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                continue
            except Exception as e:
                if "Connectivity is undefined for the null graph" in str(e):
                    error_msg = f"Graph connectivity error (null graph) for {place_name} (GEOID: {geoid}) at {buffer_size_analysis}m. File path: {graph_path}"
                else:
                    error_msg = f"Unexpected graph clipping error for {place_name} (GEOID: {geoid}) at {buffer_size_analysis}m: {e}. File path: {graph_path}"
                error_counter[error_msg] += 1
                logging.warning(error_msg)
                stats.update({
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                continue

            if G_clipped.number_of_nodes() == 0:
                error_msg = f"Clipped graph is empty for {place_name} (GEOID: {geoid}) at {buffer_size_analysis}m. File path: {graph_path}"
                error_counter[error_msg] += 1
                logging.warning(error_msg)
                stats.update({
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                continue

            # Initialize crossing stats
            for hw_type in relevant_highway_types:
                stats[f'boundary_crossing_edges_{hw_type}'] = 0
                stats[f'boundary_crossing_lanes_{hw_type}'] = 0

            # Get exterior boundary
            try:
                exterior_boundary = get_exterior_boundary(polygon_buffered_analysis)
            except ValueError as ve:
                error_msg = (
                    f"Unsupported geometry type for {place_name}, GEOID={geoid}, "
                    f"buffer={buffer_size_analysis}m: {ve}. File path: {graph_path}"
                )
                error_counter[error_msg] += 1
                logging.error(error_msg)
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                continue

            # Check boundary-crossing edges
            if not gdf_edges_full.empty:
                if 'highway' not in gdf_edges_full.columns:
                    empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                    empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                else:
                    boundary_edges = gdf_edges_full[gdf_edges_full.geometry.crosses(exterior_boundary)]
                    if 'highway' in boundary_edges.columns:
                        filtered_boundary_edges = boundary_edges[boundary_edges['highway'].apply(is_relevant_road)]
                    else:
                        filtered_boundary_edges = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')

                    if filtered_boundary_edges.empty:
                        empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                        empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                    else:
                        filtered_boundary_edges.to_csv(
                            os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"),
                            index=False
                        )
                        for _, row in filtered_boundary_edges.iterrows():
                            hw = row['highway']
                            lane_val = row.get('lanes', None)
                            if lane_val is None or (isinstance(lane_val, float) and pd.isna(lane_val)):
                                lane_count = 1
                            else:
                                if isinstance(lane_val, (list, np.ndarray)):
                                    if len(lane_val) > 0:
                                        lane_val = lane_val[0]
                                    else:
                                        lane_val = np.nan
                                try:
                                    lane_count = int(lane_val)
                                except:
                                    lane_count = 1

                            if isinstance(hw, str) and hw.strip().startswith('[') and hw.strip().endswith(']'):
                                try:
                                    hw_list = ast.literal_eval(hw)
                                except Exception:
                                    hw_list = [hw]
                            elif isinstance(hw, str):
                                hw_list = [hw]
                            elif isinstance(hw, list):
                                hw_list = hw
                            else:
                                hw_list = [str(hw)]

                            for hw_type_str in hw_list:
                                hw_type_sanitized = sanitize_hw_type(hw_type_str)
                                if hw_type_sanitized in relevant_highway_types:
                                    stats[f'boundary_crossing_edges_{hw_type_sanitized}'] += 1
                                    stats[f'boundary_crossing_lanes_{hw_type_sanitized}'] += lane_count
            else:
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)

            # Create plots if needed
            if CREATE_PLOTS:
                motorway_count = stats.get('boundary_crossing_edges_motorway', 0)
                trunk_count = stats.get('boundary_crossing_edges_trunk', 0)
                primary_count = stats.get('boundary_crossing_edges_primary', 0)
                secondary_count = stats.get('boundary_crossing_edges_secondary', 0)
                tertiary_count = stats.get('boundary_crossing_edges_tertiary', 0)

                motorway_lanes = stats.get('boundary_crossing_lanes_motorway', 0)
                trunk_lanes = stats.get('boundary_crossing_lanes_trunk', 0)
                primary_lanes = stats.get('boundary_crossing_lanes_primary', 0)
                secondary_lanes = stats.get('boundary_crossing_lanes_secondary', 0)
                tertiary_lanes = stats.get('boundary_crossing_lanes_tertiary', 0)

                fig, ax = plt.subplots(figsize=(10, 10))
                gpd.GeoSeries([polygon_buffered_analysis]).plot(
                    ax=ax, facecolor='none', edgecolor='black', lw=2
                )

                try:
                    gdf_edges_clipped = ox.graph_to_gdfs(G_clipped, nodes=False, edges=True)
                except Exception as e:
                    error_msg = f"Error converting clipped graph to GeoDataFrame for {place_name}, GEOID={geoid}: {e}. File path: {graph_path}"
                    error_counter[error_msg] += 1
                    logging.error(error_msg)
                    gdf_edges_clipped = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')

                if not gdf_edges_clipped.empty:
                    gdf_edges_clipped.plot(ax=ax, linewidth=1, color='gray')
                    for hw_type_key, hw_color in color_map.items():
                        def matches_hw_type(hw):
                            if isinstance(hw, str) and hw.strip().startswith('[') and hw.strip().endswith(']'):
                                try:
                                    hw_list = ast.literal_eval(hw)
                                except:
                                    hw_list = [hw]
                            elif isinstance(hw, str):
                                hw_list = [hw]
                            elif isinstance(hw, list):
                                hw_list = hw
                            else:
                                hw_list = [str(hw)]
                            return hw_type_key in hw_list

                        if 'highway' in gdf_edges_clipped.columns:
                            hw_edges = gdf_edges_clipped[gdf_edges_clipped['highway'].apply(matches_hw_type)]
                        else:
                            hw_edges = gdf_edges_clipped.iloc[0:0]

                        if not hw_edges.empty:
                            hw_edges.plot(ax=ax, linewidth=1.5, color=hw_color)

                    legend_handles = [
                        Line2D([0], [0], color='gray', lw=2, label='Other roads'),
                        Line2D([0], [0], color='red', lw=2, label='Motorway'),
                        Line2D([0], [0], color='orange', lw=2, label='Trunk'),
                        Line2D([0], [0], color='green', lw=2, label='Primary'),
                        Line2D([0], [0], color='purple', lw=2, label='Secondary'),
                        Line2D([0], [0], color='blue', lw=2, label='Tertiary'),
                        Line2D([0], [0], color='lightgray', lw=2, label='Residential'),
                    ]

                    boundary_edges_relevant = gdf_edges_full[
                        gdf_edges_full.geometry.crosses(exterior_boundary) &
                        gdf_edges_full['highway'].apply(is_relevant_road)
                    ]
                    intersection_points = []
                    for geom in boundary_edges_relevant.geometry:
                        try:
                            intersection = geom.intersection(exterior_boundary)
                            points = extract_points(intersection)
                            intersection_points.extend(points)
                        except Exception as e:
                            error_msg = (
                                f"Error processing intersection points for {place_name}, "
                                f"GEOID={geoid}, buffer={buffer_size_analysis}m: {e}. File path: {graph_path}"
                            )
                            error_counter[error_msg] += 1
                            logging.error(error_msg)
                            continue

                    if intersection_points:
                        intersections_gdf = gpd.GeoDataFrame(geometry=intersection_points, crs='EPSG:4326')
                        intersections_gdf.plot(
                            ax=ax,
                            markersize=100,
                            color='purple',
                            marker='o',
                            label='Boundary Intersections'
                        )
                        legend_handles.append(
                            Line2D([0], [0], marker='o', color='purple',
                                   label='Boundary Intersections',
                                   markersize=10, linestyle='None')
                        )

                    ax.legend(handles=legend_handles)

                ax.text(
                    0.05, 0.95,
                    f"Boundary Crossing Roads:\n"
                    f"Motorway: {motorway_count} roads, {motorway_lanes} lanes\n"
                    f"Trunk: {trunk_count} roads, {trunk_lanes} lanes\n"
                    f"Primary: {primary_count} roads, {primary_lanes} lanes\n"
                    f"Secondary: {secondary_count} roads, {secondary_lanes} lanes\n"
                    f"Tertiary: {tertiary_count} roads, {tertiary_lanes} lanes",
                    transform=ax.transAxes,
                    fontsize=12, verticalalignment='top',
                    bbox=dict(facecolor='white', alpha=0.7)
                )

                plot_filename = f"{sanitize_filename(place_name)}_{geoid}_visualization_{buffer_size_analysis}m.png"
                plt.tight_layout()
                plt.savefig(os.path.join(plots_output_dir, plot_filename))
                plt.close(fig)

            stats_list.append(stats)

    except TimeoutError:
        print(">>> Timeout occurred...")
        place_name = place.get('NAME', 'Unknown')
        geoid = place.get('GEOID', 'Unknown')
        state_name = place.get('STATE_NAME', 'Unknown')
        statefp = place.get('STATEFP', 'Unknown')
        timeout_msg = f"Timeout after {TIMEOUT_SECONDS}s for {place_name} (GEOID: {geoid}) in state {state_name} (STATEFP: {statefp})"
        error_counter[timeout_msg] += 1
        logging.error(timeout_msg)
        place_data = place.drop(labels='geometry', errors='ignore').to_dict()
        for buffer_size in buffer_sizes_analysis:
            fallback_stats = place_data.copy()
            fallback_stats.update({
                'buffer_distance': buffer_size,
            })
            # Try to construct CSV output path even on timeout
            sanitized_name = sanitize_filename(place_name)
            place_folder_name = f"{sanitized_name}_{geoid}"
            csv_output_dir = os.path.join(csv_dir, state_name, place_folder_name, 'csv')
            os.makedirs(csv_output_dir, exist_ok=True)
            empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
            empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size}m.csv"), index=False)
            stats_list.append(fallback_stats)

    except Exception as e:
        print(">>> Unexpected error in process_place...")
        place_name = place.get('NAME', 'Unknown')
        geoid = place.get('GEOID', 'Unknown')
        state_name = place.get('STATE_NAME', 'Unknown')
        statefp = place.get('STATEFP', 'Unknown')
        error_msg = (
            f"Unexpected error for {place_name} (GEOID: {geoid}) "
            f"in state {state_name} (STATEFP: {statefp}): {e}\n{traceback.format_exc()}"
        )
        error_counter[error_msg] += 1
        logging.error(error_msg)
        for buffer_size in buffer_sizes_analysis:
            fallback_stats = place.drop(labels='geometry', errors='ignore').to_dict()
            fallback_stats.update({
                'buffer_distance': buffer_size,
            })
            stats_list.append(fallback_stats)

    finally:
        signal.alarm(0)

    aggregated_errors = [f"{msg} (Occurred {count} times)" for msg, count in error_counter.items()]
    return stats_list, aggregated_errors

################################################################################
#                                    MAIN LOGIC                                 #
################################################################################
print(">>> Starting main logic...")

logging.basicConfig(
    filename='processing_us_graphs.log',
    level=logging.INFO,
    format='%(asctime)s:%(levelname)s:%(message)s'
)

print(">>> Finding state directories...")
all_state_dirs = [d for d in glob.glob(os.path.join(US_CENSUS_PLACES_DIR, '*')) if os.path.isdir(d)]
if not all_state_dirs:
    error_msg = "No state directories found in US census places directory."
    logging.error(error_msg)
    print(error_msg)
    exit()

print(">>> Reading US census place shapefiles...")
gdf_all_places = gpd.GeoDataFrame()

for state_dir in all_state_dirs:
    # Extract state name and FIPS code from directory name
    base = os.path.basename(state_dir)
    if '_' not in base:
        logging.warning(f"Skipping directory {base} - doesn't match expected format (StateName_FIPS)")
        continue
    
    state_name_part, statefp_part = base.rsplit('_', 1)
    
    # Skip if we're filtering by states and this isn't in the list
    if SELECTED_STATES and state_name_part not in SELECTED_STATES:
        continue
    
    # Find all shapefiles in the state directory
    for shp_file in glob.glob(os.path.join(state_dir, '*.shp')):
        try:
            # Read the shapefile
            gdf = gpd.read_file(shp_file)
            
            # Ensure CRS is set to EPSG:4326
            if gdf.crs is None:
                gdf.set_crs(epsg=4326, inplace=True)
            else:
                gdf = gdf.to_crs(epsg=4326)
            
            # Add state information to the dataframe
            gdf['STATEFP'] = statefp_part
            gdf['STATE_NAME'] = state_name_part
            
            # Append to the main dataframe
            gdf_all_places = pd.concat([gdf_all_places, gdf], ignore_index=True)
            
            print(f"Loaded {len(gdf)} places from {os.path.basename(shp_file)}")
            
        except Exception as e:
            error_msg = f"Error reading shapefile {shp_file}: {e}"
            logging.error(error_msg)
            print(error_msg)

if gdf_all_places.empty:
    error_msg = "No census places found after reading all shapefiles."
    logging.error(error_msg)
    print(error_msg)
    exit()

# Check required columns
required_columns = ['NAME', 'GEOID', 'STATE_NAME', 'STATEFP', 'geometry']
missing_columns = [col for col in required_columns if col not in gdf_all_places.columns]
if missing_columns:
    # Try to find alternative column names
    if 'NAME' not in gdf_all_places.columns and 'NAMELSAD' in gdf_all_places.columns:
        gdf_all_places['NAME'] = gdf_all_places['NAMELSAD']
    
    # Re-check after potential fixes
    missing_columns = [col for col in required_columns if col not in gdf_all_places.columns]
    if missing_columns:
        error_msg = f"Missing required columns in shapefiles: {missing_columns}"
        logging.error(error_msg)
        print(error_msg)
        print(f"Available columns: {list(gdf_all_places.columns)}")
        exit()

print(f">>> Total US census places loaded: {len(gdf_all_places)}")
print(">>> Head of the US places DataFrame before processing:")
print(gdf_all_places.head())

print(">>> Starting parallel processing for US places...")
process_func = partial(
    process_place,
    graphml_dir=GRAPHML_OUTPUT_DIR_US,
    plots_dir=PLOTS_OUTPUT_DIR_US,
    csv_dir=CSV_OUTPUT_DIR_US,
    buffer_sizes_analysis=BUFFER_SIZES_ANALYSIS_METERS
)

network_stats_list = []
aggregated_error_messages = []

with ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
    futures = {
        executor.submit(process_func, place): place
        for idx, place in gdf_all_places.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Places"):
        place = futures[future]
        place_name = place.get('NAME', 'Unknown')
        geoid = place.get('GEOID', 'Unknown')
        state_name = place.get('STATE_NAME', 'Unknown')
        statefp = place.get('STATEFP', 'Unknown')
        try:
            stats, errors = future.result()
            if stats:
                network_stats_list.extend(stats)
            if errors:
                aggregated_error_messages.extend(errors)
        except Exception as e:
            general_error_msg = (
                f"Unexpected error processing place {place_name} "
                f"(GEOID: {geoid}) in state {state_name} (STATEFP: {statefp}): {e}\n{traceback.format_exc()}"
            )
            aggregated_error_messages.append(general_error_msg)
            logging.error(general_error_msg)
            for buffer_size in BUFFER_SIZES_ANALYSIS_METERS:
                fallback_stats = place.drop(labels='geometry', errors='ignore').to_dict()
                fallback_stats.update({
                    'buffer_distance': buffer_size,
                })
                network_stats_list.append(fallback_stats)

print(">>> Converting network stats to a DataFrame...")
try:
    if not network_stats_list:
        print("Warning: No network statistics were collected. Creating empty DataFrame.")
        network_stats_df = pd.DataFrame()
    else:
        network_stats_df = pd.DataFrame(network_stats_list)
        print(f"Successfully created DataFrame with {len(network_stats_df)} rows and {len(network_stats_df.columns)} columns.")
except Exception as e:
    error_msg = f"Error converting network stats to DataFrame: {e}"
    logging.error(error_msg)
    print(f"Error details: {error_msg}")
    print("Creating fallback empty DataFrame...")
    network_stats_df = pd.DataFrame()

print(">>> Saving network statistics to CSV...")
try:
    if network_stats_df.empty:
        print("Warning: DataFrame is empty. Creating minimal CSV with headers only.")
        # Create a minimal DataFrame with expected columns
        minimal_df = pd.DataFrame(columns=['GEOID', 'NAME', 'STATE_NAME', 'STATEFP', 'buffer_distance'])
        minimal_df.to_csv(FINAL_CSV_OUTPUT_US, index=False)
        print(f"Empty CSV saved to {FINAL_CSV_OUTPUT_US}")
    else:
        network_stats_df.to_csv(FINAL_CSV_OUTPUT_US, index=False)
        print(f"Network statistics saved to {FINAL_CSV_OUTPUT_US}")
        logging.info(f"Network statistics saved to {FINAL_CSV_OUTPUT_US}")
except Exception as e:
    error_msg = f"Error saving network statistics to CSV: {e}"
    logging.error(error_msg)
    print(f"CSV save error details: {error_msg}")
    print("Attempting to save a diagnostic CSV...")
    try:
        # Try to save just the first few rows to diagnose the issue
        if not network_stats_df.empty:
            sample_df = network_stats_df.head(10)
            sample_path = FINAL_CSV_OUTPUT_US.replace('.csv', '_sample.csv')
            sample_df.to_csv(sample_path, index=False)
            print(f"Sample CSV saved to {sample_path}")
    except Exception as e2:
        print(f"Even sample CSV failed: {e2}")

if aggregated_error_messages:
    print("\n### Errors Encountered During Processing ###\n")
    for msg in aggregated_error_messages:
        print(msg)
    print(f"\nTotal Errors: {len(aggregated_error_messages)}")
    logging.info(f"Total Errors: {len(aggregated_error_messages)}")
else:
    print("\nAll places processed successfully without errors.")
    logging.info("All places processed successfully without errors.")

## Canada

In [ ]:
import os
import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use a non-interactive backend
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, MultiLineString, Point
from matplotlib.lines import Line2D
import networkx as nx
import numpy as np
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial
from tqdm import tqdm
import logging
import glob
import signal
import ast
import multiprocessing
from collections import defaultdict

################################################################################
#                            CONFIGURATION / SETTINGS                          #
################################################################################
print(">>> Setting up configuration...")
NUM_CORES = 200
BUFFER_SIZES_ANALYSIS_METERS = [500]

GRAPHML_OUTPUT_DIR_CANADA = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'osmnx_road_network', '2024', 'Canada', 'Canada')
PLOTS_OUTPUT_DIR_CANADA = os.path.join(WRI_PROJECT_ROOT, 'data', 'infrastructure', 'intermediate', 'road_network', '2024')
CSV_OUTPUT_DIR_CANADA = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'osmnx_road_network', '2024', 'Canada')
CANADA_CDP_SHAPEFILE = os.path.join(WRI_PROJECT_ROOT, 'data', 'infrastructure', 'intermediate', 'road_network', '2024', 'canada_designated_places_equivalent', 'canada_cdp_equivalent.shp')
FINAL_CSV_OUTPUT_CANADA = os.path.join(WRI_PROJECT_ROOT, 'data', 'infrastructure', 'intermediate', 'road_network', '2024', 'canada_roadnetwork_statistics.csv')

CREATE_PLOTS = True
TIMEOUT_SECONDS = 600

os.makedirs(GRAPHML_OUTPUT_DIR_CANADA, exist_ok=True)
os.makedirs(PLOTS_OUTPUT_DIR_CANADA, exist_ok=True)
os.makedirs(CSV_OUTPUT_DIR_CANADA, exist_ok=True)
print(">>> Configuration complete.")

################################################################################
#                                HELPER FUNCTIONS                              #
################################################################################
def sanitize_filename(name):
    return "".join(c if c.isalnum() else "_" for c in name).strip("_")

def get_exterior_boundary(polygon):
    if isinstance(polygon, Polygon):
        return polygon.exterior
    elif isinstance(polygon, MultiPolygon):
        return MultiLineString([p.exterior for p in polygon.geoms])
    else:
        raise ValueError("Unsupported geometry type for exterior boundary extraction.")

def sanitize_hw_type(hw_type):
    return hw_type.replace(' ', '_').replace('[', '').replace(']', '').replace(',', '').replace("'", "")

def timeout_handler(signum, frame):
    raise TimeoutError("Operation timed out inside worker process.")

def extract_points(geometry):
    points = []
    if geometry.is_empty:
        return points
    elif geometry.geom_type == 'Point':
        points.append(geometry)
    elif geometry.geom_type == 'MultiPoint':
        points.extend(list(geometry.geoms))
    elif geometry.geom_type == 'GeometryCollection':
        for geom in geometry.geoms:
            if geom.geom_type == 'Point':
                points.append(geom)
    elif geometry.geom_type == 'LineString':
        for coord in geometry.coords:
            points.append(Point(coord))
    return points

def find_graphml_file(graphml_dir, dpluid):
    """
    Find the GraphML file for a given DPLUID.
    
    Args:
        graphml_dir: Base directory containing GraphML files
        dpluid: The DPLUID to search for
    
    Returns:
        tuple: (graph_path, place_folder_name) if found, (None, None) if not found
    """
    # Pattern 1: Look for folders matching Place_*_DPLUID
    folder_pattern = os.path.join(graphml_dir, f"Place_*_{dpluid}")
    matched_folders = glob.glob(folder_pattern)
    
    if not matched_folders:
        # Pattern 2: Alternative pattern if needed
        folder_pattern_alt = os.path.join(graphml_dir, f"*_{dpluid}")
        matched_folders = glob.glob(folder_pattern_alt)
    
    if len(matched_folders) == 1:
        folder_path = matched_folders[0]
        folder_name = os.path.basename(folder_path)
        
        # Look for GraphML file in the folder
        # First try the exact folder name as GraphML filename
        graphml_path = os.path.join(folder_path, f"{folder_name}.graphml")
        if os.path.exists(graphml_path):
            return graphml_path, folder_name
        
        # If that doesn't exist, look for any .graphml file in the folder
        graphml_pattern = os.path.join(folder_path, "*.graphml")
        graphml_files = glob.glob(graphml_pattern)
        
        if len(graphml_files) == 1:
            return graphml_files[0], folder_name
        elif len(graphml_files) > 1:
            # Multiple GraphML files found, return the first one
            return graphml_files[0], folder_name
        else:
            # No GraphML files found in the folder
            return None, None
    
    elif len(matched_folders) > 1:
        # Multiple folders found, this is an error condition
        return "multiple", None
    else:
        # No folders found
        return None, None

################################################################################
#                              PROCESSING FUNCTION                             #
################################################################################
def process_place(place, graphml_dir, plots_dir, csv_dir, buffer_sizes_analysis):
    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(TIMEOUT_SECONDS)

    relevant_highway_types = ['motorway', 'trunk', 'primary', 'secondary', 'tertiary']
    color_map = {
        'motorway': 'red',
        'trunk': 'orange',
        'primary': 'green',
        'secondary': 'purple',
        'tertiary': 'blue',
        'residential': 'lightgray'
    }

    def is_relevant_road(hw):
        if isinstance(hw, str) and hw.strip().startswith('[') and hw.strip().endswith(']'):
            try:
                hw_list = ast.literal_eval(hw)
            except Exception:
                hw_list = [hw]
        elif isinstance(hw, str):
            hw_list = [hw]
        elif isinstance(hw, list):
            hw_list = hw
        else:
            hw_list = [str(hw)]
        return any(sanitize_hw_type(h) in relevant_highway_types for h in hw_list)

    stats_list = []
    error_counter = defaultdict(int)

    try:
        place_name = place['DPLNAME']
        dpluid = place['DPLUID']
        polygon = place.geometry

        # Find the correct GraphML file using the updated function
        graph_path, place_folder_name = find_graphml_file(graphml_dir, dpluid)

        if graph_path is None:
            error_msg = f"GraphML file not found for {place_name} (DPLUID: {dpluid})."
            error_counter[error_msg] += 1
            logging.error(error_msg)
            return stats_list, list(error_counter.keys())
        elif graph_path == "multiple":
            error_msg = f"Multiple GraphML folders found for {place_name} (DPLUID: {dpluid})."
            error_counter[error_msg] += 1
            logging.error(error_msg)
            return stats_list, list(error_counter.keys())
        
        plots_output_dir = os.path.join(plots_dir, 'Canada', place_folder_name)
        os.makedirs(plots_output_dir, exist_ok=True)

        # Use the new CSV output directory structure
        csv_output_dir = os.path.join(csv_dir, place_folder_name, 'csv')
        os.makedirs(csv_output_dir, exist_ok=True)

        # Load graph
        try:
            G = ox.load_graphml(graph_path)
        except Exception as e:
            error_msg = f"Failed to load GraphML file for {place_name} (DPLUID: {dpluid}): {e}"
            error_counter[error_msg] += 1
            logging.error(error_msg)
            return stats_list, list(error_counter.keys())

        # Check if graph has any edges before proceeding
        if G.number_of_edges() == 0:
            error_msg = f"Graph contains no edges for {place_name} (DPLUID: {dpluid})."
            error_counter[error_msg] += 1
            logging.warning(error_msg)
            for buffer_size_analysis in buffer_sizes_analysis:
                stats = place.drop(labels='geometry', errors='ignore').to_dict()
                stats.update({
                    'buffer_distance': buffer_size_analysis,
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
            return stats_list, list(error_counter.keys())

        # Convert to GeoDataFrames with error handling
        try:
            gdf_nodes_full, gdf_edges_full = ox.graph_to_gdfs(G, nodes=True, edges=True)
        except ValueError as e:
            if "Graph contains no edges" in str(e):
                error_msg = f"Graph contains no edges for {place_name} (DPLUID: {dpluid})."
                error_counter[error_msg] += 1
                logging.warning(error_msg)
                for buffer_size_analysis in buffer_sizes_analysis:
                    stats = place.drop(labels='geometry', errors='ignore').to_dict()
                    stats.update({
                        'buffer_distance': buffer_size_analysis,
                    })
                    stats_list.append(stats)
                    empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                    empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                return stats_list, list(error_counter.keys())
            else:
                raise e
        except Exception as e:
            error_msg = f"Failed to convert graph to GeoDataFrames for {place_name} (DPLUID: {dpluid}): {e}"
            error_counter[error_msg] += 1
            logging.error(error_msg)
            return stats_list, list(error_counter.keys())

        if gdf_edges_full is None or gdf_edges_full.empty:
            error_msg = f"Graph contains no edges for {place_name} (DPLUID: {dpluid})."
            error_counter[error_msg] += 1
            logging.warning(error_msg)
            for buffer_size_analysis in buffer_sizes_analysis:
                stats = place.drop(labels='geometry', errors='ignore').to_dict()
                stats.update({
                    'buffer_distance': buffer_size_analysis,
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
            return stats_list, list(error_counter.keys())

        # Ensure CRS is set to EPSG:4326
        if gdf_edges_full.crs is None:
            gdf_edges_full.set_crs(epsg=4326, inplace=True)
        elif gdf_edges_full.crs.to_string() != 'EPSG:4326':
            gdf_edges_full = gdf_edges_full.to_crs(epsg=4326)

        place_data = place.drop(labels='geometry', errors='ignore').to_dict()

        for buffer_size_analysis in buffer_sizes_analysis:
            stats = place_data.copy()
            stats['buffer_distance'] = buffer_size_analysis

            # Buffer the polygon
            polygon_series = gpd.GeoSeries([polygon], crs='EPSG:4326')
            utm_crs = polygon_series.estimate_utm_crs()
            polygon_utm = polygon_series.to_crs(utm_crs)
            polygon_buffered_analysis = polygon_utm.buffer(buffer_size_analysis).to_crs('EPSG:4326').iloc[0]

            # Clip the graph with additional error handling
            try:
                # Check if graph is empty before clipping
                if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
                    raise ValueError("Empty graph cannot be clipped")
                
                G_clipped = ox.truncate.truncate_graph_polygon(
                    G,
                    polygon_buffered_analysis,
                    truncate_by_edge=True
                )
            except ValueError as e:
                if "Empty graph cannot be clipped" in str(e):
                    error_msg = f"Cannot clip empty graph for {place_name} (DPLUID: {dpluid}) at {buffer_size_analysis}m."
                elif "No graph nodes found within buffer" in str(e) or "No graph nodes remain" in str(e):
                    error_msg = f"No graph nodes found within buffer for {place_name} (DPLUID: {dpluid}) at {buffer_size_analysis}m."
                else:
                    error_msg = f"Graph clipping failed for {place_name} (DPLUID: {dpluid}) at {buffer_size_analysis}m: {e}"
                error_counter[error_msg] += 1
                logging.warning(error_msg)
                stats.update({
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                continue
            except Exception as e:
                if "Connectivity is undefined for the null graph" in str(e):
                    error_msg = f"Graph connectivity error (null graph) for {place_name} (DPLUID: {dpluid}) at {buffer_size_analysis}m."
                else:
                    error_msg = f"Unexpected graph clipping error for {place_name} (DPLUID: {dpluid}) at {buffer_size_analysis}m: {e}"
                error_counter[error_msg] += 1
                logging.warning(error_msg)
                stats.update({
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                continue

            if G_clipped.number_of_nodes() == 0:
                error_msg = f"Clipped graph is empty for {place_name} (DPLUID: {dpluid}) at {buffer_size_analysis}m."
                error_counter[error_msg] += 1
                logging.warning(error_msg)
                stats.update({
                })
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                continue



            # Initialize crossing stats
            for hw_type in relevant_highway_types:
                stats[f'boundary_crossing_edges_{hw_type}'] = 0
                stats[f'boundary_crossing_lanes_{hw_type}'] = 0

            # Get exterior boundary
            try:
                exterior_boundary = get_exterior_boundary(polygon_buffered_analysis)
            except ValueError as ve:
                error_msg = (
                    f"Unsupported geometry type for {place_name}, DPLUID={dpluid}, "
                    f"buffer={buffer_size_analysis}m: {ve}"
                )
                error_counter[error_msg] += 1
                logging.error(error_msg)
                stats_list.append(stats)
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                continue

            # Check boundary-crossing edges
            if not gdf_edges_full.empty:
                if 'highway' not in gdf_edges_full.columns:
                    empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                    empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                else:
                    boundary_edges = gdf_edges_full[gdf_edges_full.geometry.crosses(exterior_boundary)]
                    if 'highway' in boundary_edges.columns:
                        filtered_boundary_edges = boundary_edges[boundary_edges['highway'].apply(is_relevant_road)]
                    else:
                        filtered_boundary_edges = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')

                    if filtered_boundary_edges.empty:
                        empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                        empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)
                    else:
                        filtered_boundary_edges.to_csv(
                            os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"),
                            index=False
                        )
                        for _, row in filtered_boundary_edges.iterrows():
                            hw = row['highway']
                            lane_val = row.get('lanes', None)
                            if lane_val is None or (isinstance(lane_val, float) and pd.isna(lane_val)):
                                lane_count = 1
                            else:
                                if isinstance(lane_val, (list, np.ndarray)):
                                    if len(lane_val) > 0:
                                        lane_val = lane_val[0]
                                    else:
                                        lane_val = np.nan
                                try:
                                    lane_count = int(lane_val)
                                except:
                                    lane_count = 1

                            if isinstance(hw, str) and hw.strip().startswith('[') and hw.strip().endswith(']'):
                                try:
                                    hw_list = ast.literal_eval(hw)
                                except Exception:
                                    hw_list = [hw]
                            elif isinstance(hw, str):
                                hw_list = [hw]
                            elif isinstance(hw, list):
                                hw_list = hw
                            else:
                                hw_list = [str(hw)]

                            for hw_type_str in hw_list:
                                hw_type_sanitized = sanitize_hw_type(hw_type_str)
                                if hw_type_sanitized in relevant_highway_types:
                                    stats[f'boundary_crossing_edges_{hw_type_sanitized}'] += 1
                                    stats[f'boundary_crossing_lanes_{hw_type_sanitized}'] += lane_count
            else:
                empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
                empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size_analysis}m.csv"), index=False)

            # Create plots if needed
            if CREATE_PLOTS:
                motorway_count = stats.get('boundary_crossing_edges_motorway', 0)
                trunk_count = stats.get('boundary_crossing_edges_trunk', 0)
                primary_count = stats.get('boundary_crossing_edges_primary', 0)
                secondary_count = stats.get('boundary_crossing_edges_secondary', 0)
                tertiary_count = stats.get('boundary_crossing_edges_tertiary', 0)

                motorway_lanes = stats.get('boundary_crossing_lanes_motorway', 0)
                trunk_lanes = stats.get('boundary_crossing_lanes_trunk', 0)
                primary_lanes = stats.get('boundary_crossing_lanes_primary', 0)
                secondary_lanes = stats.get('boundary_crossing_lanes_secondary', 0)
                tertiary_lanes = stats.get('boundary_crossing_lanes_tertiary', 0)

                fig, ax = plt.subplots(figsize=(10, 10))
                gpd.GeoSeries([polygon_buffered_analysis]).plot(
                    ax=ax, facecolor='none', edgecolor='black', lw=2
                )

                try:
                    gdf_edges_clipped = ox.graph_to_gdfs(G_clipped, nodes=False, edges=True)
                except Exception as e:
                    error_msg = f"Error converting clipped graph to GeoDataFrame for {place_name}, DPLUID={dpluid}: {e}"
                    error_counter[error_msg] += 1
                    logging.error(error_msg)
                    gdf_edges_clipped = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')

                if not gdf_edges_clipped.empty:
                    gdf_edges_clipped.plot(ax=ax, linewidth=1, color='gray')
                    for hw_type_key, hw_color in color_map.items():
                        def matches_hw_type(hw):
                            if isinstance(hw, str) and hw.strip().startswith('[') and hw.strip().endswith(']'):
                                try:
                                    hw_list = ast.literal_eval(hw)
                                except:
                                    hw_list = [hw]
                            elif isinstance(hw, str):
                                hw_list = [hw]
                            elif isinstance(hw, list):
                                hw_list = hw
                            else:
                                hw_list = [str(hw)]
                            return hw_type_key in hw_list

                        if 'highway' in gdf_edges_clipped.columns:
                            hw_edges = gdf_edges_clipped[gdf_edges_clipped['highway'].apply(matches_hw_type)]
                        else:
                            hw_edges = gdf_edges_clipped.iloc[0:0]

                        if not hw_edges.empty:
                            hw_edges.plot(ax=ax, linewidth=1.5, color=hw_color)

                    legend_handles = [
                        Line2D([0], [0], color='gray', lw=2, label='Other roads'),
                        Line2D([0], [0], color='red', lw=2, label='Motorway'),
                        Line2D([0], [0], color='orange', lw=2, label='Trunk'),
                        Line2D([0], [0], color='green', lw=2, label='Primary'),
                        Line2D([0], [0], color='purple', lw=2, label='Secondary'),
                        Line2D([0], [0], color='blue', lw=2, label='Tertiary'),
                        Line2D([0], [0], color='lightgray', lw=2, label='Residential'),
                    ]

                    boundary_edges_relevant = gdf_edges_full[
                        gdf_edges_full.geometry.crosses(exterior_boundary) &
                        gdf_edges_full['highway'].apply(is_relevant_road)
                    ]
                    intersection_points = []
                    for geom in boundary_edges_relevant.geometry:
                        try:
                            intersection = geom.intersection(exterior_boundary)
                            points = extract_points(intersection)
                            intersection_points.extend(points)
                        except Exception as e:
                            error_msg = (
                                f"Error processing intersection points for {place_name}, "
                                f"DPLUID={dpluid}, buffer={buffer_size_analysis}m: {e}"
                            )
                            error_counter[error_msg] += 1
                            logging.error(error_msg)
                            continue

                    if intersection_points:
                        intersections_gdf = gpd.GeoDataFrame(geometry=intersection_points, crs='EPSG:4326')
                        intersections_gdf.plot(
                            ax=ax,
                            markersize=100,
                            color='purple',
                            marker='o',
                            label='Boundary Intersections'
                        )
                        legend_handles.append(
                            Line2D([0], [0], marker='o', color='purple',
                                   label='Boundary Intersections',
                                   markersize=10, linestyle='None')
                        )

                    ax.legend(handles=legend_handles)

                ax.text(
                    0.05, 0.95,
                    f"Boundary Crossing Roads:\n"
                    f"Motorway: {motorway_count} roads, {motorway_lanes} lanes\n"
                    f"Trunk: {trunk_count} roads, {trunk_lanes} lanes\n"
                    f"Primary: {primary_count} roads, {primary_lanes} lanes\n"
                    f"Secondary: {secondary_count} roads, {secondary_lanes} lanes\n"
                    f"Tertiary: {tertiary_count} roads, {tertiary_lanes} lanes",
                    transform=ax.transAxes,
                    fontsize=12, verticalalignment='top',
                    bbox=dict(facecolor='white', alpha=0.7)
                )

                plot_filename = f"{sanitize_filename(place_name)}_{dpluid}_visualization_{buffer_size_analysis}m.png"
                plt.tight_layout()
                plt.savefig(os.path.join(plots_output_dir, plot_filename))
                plt.close(fig)

            stats_list.append(stats)

    except TimeoutError:
        print(">>> Timeout occurred...")  # Keeping as an error indicator
        place_name = place.get('DPLNAME', 'Unknown')
        dpluid = place.get('DPLUID', 'Unknown')
        pruid = place.get('PRUID', 'Unknown')
        timeout_msg = f"Timeout after {TIMEOUT_SECONDS}s for {place_name} (DPLUID: {dpluid}) in PRUID {pruid}"
        error_counter[timeout_msg] += 1
        logging.error(timeout_msg)
        place_data = place.drop(labels='geometry', errors='ignore').to_dict()
        for buffer_size in buffer_sizes_analysis:
            fallback_stats = place_data.copy()
            fallback_stats.update({
                'buffer_distance': buffer_size,
            })
            # Try to find the folder for CSV output even on timeout
            _, place_folder_name = find_graphml_file(graphml_dir, dpluid)
            if place_folder_name:
                csv_output_dir = os.path.join(csv_dir, place_folder_name, 'csv')
            else:
                csv_output_dir = os.path.join(csv_dir, f"Place_fallback_{dpluid}", 'csv')
            os.makedirs(csv_output_dir, exist_ok=True)
            empty_gdf = gpd.GeoDataFrame(columns=['highway', 'geometry'], crs='EPSG:4326')
            empty_gdf.to_csv(os.path.join(csv_output_dir, f"boundary_crossing_roads_{buffer_size}m.csv"), index=False)
            stats_list.append(fallback_stats)

    except Exception as e:
        print(">>> Unexpected error in process_place...")  # Keeping as an error indicator
        place_name = place.get('DPLNAME', 'Unknown')
        dpluid = place.get('DPLUID', 'Unknown')
        pruid = place.get('PRUID', 'Unknown')
        error_msg = (
            f"Unexpected error for {place_name} (DPLUID: {dpluid}) "
            f"in PRUID {pruid}: {e}\n{traceback.format_exc()}"
        )
        error_counter[error_msg] += 1
        logging.error(error_msg)
        for buffer_size in buffer_sizes_analysis:
            fallback_stats = place.drop(labels='geometry', errors='ignore').to_dict()
            fallback_stats.update({
                'buffer_distance': buffer_size,
            })
            stats_list.append(fallback_stats)

    finally:
        signal.alarm(0)

    aggregated_errors = [f"{msg} (Occurred {count} times)" for msg, count in error_counter.items()]
    return stats_list, aggregated_errors

################################################################################
#                                    MAIN LOGIC                                 #
################################################################################
print(">>> Starting main logic...")

logging.basicConfig(
    filename='processing_canadian_graphs.log',
    level=logging.INFO,
    format='%(asctime)s:%(levelname)s:%(message)s'
)

print(">>> Loading Canadian designated places shapefile...")
try:
    gdf_all_places = gpd.read_file(CANADA_CDP_SHAPEFILE)
    if gdf_all_places.crs is None:
        gdf_all_places.set_crs(epsg=4326, inplace=True)
    else:
        gdf_all_places = gdf_all_places.to_crs(epsg=4326)
except Exception as e:
    error_msg = f"Error reading Canadian shapefile {CANADA_CDP_SHAPEFILE}: {e}"
    logging.error(error_msg)
    print(error_msg)
    exit()

required_columns = ['DPLNAME', 'DPLUID', 'PRUID', 'geometry']
missing_columns = [col for col in required_columns if col not in gdf_all_places.columns]
if missing_columns:
    error_msg = f"Missing required columns in shapefile: {missing_columns}"
    logging.error(error_msg)
    print(error_msg)
    exit()

print(">>> Head of the Canadian places DataFrame before processing:")
print(gdf_all_places.head())

print(">>> Starting parallel processing for Canadian places...")
process_func = partial(
    process_place,
    graphml_dir=GRAPHML_OUTPUT_DIR_CANADA,
    plots_dir=PLOTS_OUTPUT_DIR_CANADA,
    csv_dir=CSV_OUTPUT_DIR_CANADA,
    buffer_sizes_analysis=BUFFER_SIZES_ANALYSIS_METERS
)

network_stats_list = []
aggregated_error_messages = []

with ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
    futures = {
        executor.submit(process_func, place): place
        for idx, place in gdf_all_places.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Places"):
        place = futures[future]
        place_name = place.get('DPLNAME', 'Unknown')
        dpluid = place.get('DPLUID', 'Unknown')
        pruid = place.get('PRUID', 'Unknown')
        try:
            stats, errors = future.result()
            if stats:
                network_stats_list.extend(stats)
            if errors:
                aggregated_error_messages.extend(errors)
        except Exception as e:
            general_error_msg = (
                f"Unexpected error processing place {place_name} "
                f"(DPLUID: {dpluid}) in PRUID {pruid}: {e}\n{traceback.format_exc()}"
            )
            aggregated_error_messages.append(general_error_msg)
            logging.error(general_error_msg)
            for buffer_size in BUFFER_SIZES_ANALYSIS_METERS:
                fallback_stats = place.drop(labels='geometry', errors='ignore').to_dict()
                fallback_stats.update({
                    'buffer_distance': buffer_size,
                })
                network_stats_list.append(fallback_stats)

print(">>> Converting network stats to a DataFrame...")
try:
    if not network_stats_list:
        print("Warning: No network statistics were collected. Creating empty DataFrame.")
        network_stats_df = pd.DataFrame()
    else:
        network_stats_df = pd.DataFrame(network_stats_list)
        print(f"Successfully created DataFrame with {len(network_stats_df)} rows and {len(network_stats_df.columns)} columns.")
except Exception as e:
    error_msg = f"Error converting network stats to DataFrame: {e}"
    logging.error(error_msg)
    print(f"Error details: {error_msg}")
    print("Creating fallback empty DataFrame...")
    network_stats_df = pd.DataFrame()

print(">>> Saving network statistics to CSV...")
try:
    if network_stats_df.empty:
        print("Warning: DataFrame is empty. Creating minimal CSV with headers only.")
        # Create a minimal DataFrame with expected columns
        minimal_df = pd.DataFrame(columns=['DPLUID', 'DPLNAME', 'PRUID', 'buffer_distance'])
        minimal_df.to_csv(FINAL_CSV_OUTPUT_CANADA, index=False)
        print(f"Empty CSV saved to {FINAL_CSV_OUTPUT_CANADA}")
    else:
        network_stats_df.to_csv(FINAL_CSV_OUTPUT_CANADA, index=False)
        print(f"Network statistics saved to {FINAL_CSV_OUTPUT_CANADA}")
        logging.info(f"Network statistics saved to {FINAL_CSV_OUTPUT_CANADA}")
except Exception as e:
    error_msg = f"Error saving network statistics to CSV: {e}"
    logging.error(error_msg)
    print(f"CSV save error details: {error_msg}")
    print("Attempting to save a diagnostic CSV...")
    try:
        # Try to save just the first few rows to diagnose the issue
        if not network_stats_df.empty:
            sample_df = network_stats_df.head(10)
            sample_path = FINAL_CSV_OUTPUT_CANADA.replace('.csv', '_sample.csv')
            sample_df.to_csv(sample_path, index=False)
            print(f"Sample CSV saved to {sample_path}")
    except Exception as e2:
        print(f"Even sample CSV failed: {e2}")

if aggregated_error_messages:
    print("\n### Errors Encountered During Processing ###\n")
    for msg in aggregated_error_messages:
        print(msg)
    print(f"\nTotal Errors: {len(aggregated_error_messages)}")
    logging.info(f"Total Errors: {len(aggregated_error_messages)}")
else:
    print("\nAll places processed successfully without errors.")
    logging.info("All places processed successfully without errors.")